## SRP519017

**paper:** [PMID: 41060812](https://pmc.ncbi.nlm.nih.gov/articles/PMC12679999/) - The transcriptomes of hypothalamic micropunches reveal sex differences in regulatory processes across hibernation in the Arctic ground squirrel, 2025

**date, curator:** 2026-04-17, Sara Carsanaro

**notes**
* paper is about arcuate nucleus however they sampled arcuate nucleus, median eminence, pars tuberalis, and third ventricle - hypothalamus is more appropriate here

### annotation summary

In [22]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Brain - hypothalamus,UBERON:0001898,hypothalamus,perfect match


In [23]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,juvenile,UBERON:0034919,juvenile stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP519017"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|███████████████████████████████████████████| 14/14 [00:14<00:00,  1.06s/it]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX25262324,SRP519017,Illumina NovaSeq X Plus,SRS21947445,,,,,,Brain - hypothalamus,juvenile,,,,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,AGSM2024,SAMN42392262,,,,,,,,LH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2024,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX25262323,SRP519017,Illumina NovaSeq X Plus,SRS21947444,,,,,,Brain - hypothalamus,juvenile,,,,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,AGSM2018,SAMN42392261,,,,,,,,EH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2018,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX25262322,SRP519017,Illumina NovaSeq X Plus,SRS21947443,,,,,,Brain - hypothalamus,juvenile,,,,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,AGSM2008,SAMN42392260,,,,,,,,LH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2008,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX25262321,SRP519017,Illumina NovaSeq X Plus,SRS21947442,,,,,,Brain - hypothalamus,juvenile,,,,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,AGSM2007,SAMN42392259,,,,,,,,LH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2007,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX25262320,SRP519017,Illumina NovaSeq X Plus,SRS21947441,,,,,,Brain - hypothalamus,juvenile,,,,F,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,AGSF2083,SAMN42392258,,,,,,,,EH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSF2083,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX25262319,SRP519017,Illumina NovaSeq X Plus,SRS21947440,,,,,,Brain - hypothalamus,juvenile,,,,F,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,AGSF2073,SAMN42392257,,,,,,,,LH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSF2073,,,,,,TRANSCRIPTOMIC,cDNA
6,SRX25262318,SRP519017,Illumina NovaSeq X Plus,SRS21947439,,,,,,Brain - hypothalamus,juvenile,,,,F,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,AGSF2060,SAMN42392256,,,,,,,,EJ,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mR

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['Brain - hypothalamus']


In [6]:

# all
library.loc[:,'anatId'] = 'UBERON:0001898'
library.loc[:,'anatName'] = 'hypothalamus'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX25262324,SRP519017,Illumina NovaSeq X Plus,SRS21947445,UBERON:0001898,hypothalamus,,,,Brain - hypothalamus,juvenile,perfect match,not documented,,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,AGSM2024,SAMN42392262,,,,,,,,LH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2024,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX25262323,SRP519017,Illumina NovaSeq X Plus,SRS21947444,UBERON:0001898,hypothalamus,,,,Brain - hypothalamus,juvenile,perfect match,not documented,,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,AGSM2018,SAMN42392261,,,,,,,,EH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2018,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX25262322,SRP519017,Illumina NovaSeq X Plus,SRS21947443,UBERON:0001898,hypothalamus,,,,Brain - hypothalamus,juvenile,perfect match,not documented,,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,AGSM2008,SAMN42392260,,,,,,,,LH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2008,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX25262321,SRP519017,Illumina NovaSeq X Plus,SRS21947442,UBERON:0001898,hypothalamus,,,,Brain - hypothalamus,juvenile,perfect match,not documented,,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,AGSM2007,SAMN42392259,,,,,,,,LH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2007,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX25262320,SRP519017,Illumina NovaSeq X Plus,SRS21947441,UBERON:0001898,hypothalamus,,,,Brain - hypothalamus,juvenile,perfect match,not documented,,F,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,AGSF2083,SAMN42392258,,,,,,,,EH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSF2083,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX25262319,SRP519017,Illumina NovaSeq X Plus,SRS21947440,UBERON:0001898,hypothalamus,,,,Brain - hypothalamus,juvenile,perfect match,not documented,,F,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,AGSF2073,SAMN42392257,,,,,,,,LH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSF2073,,,,,,TRANSCRIPTOMIC,cDNA
6,SRX25262318,SRP

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['juvenile']


In [8]:
# all
library.loc[:,'stageId'] = 'UBERON:0034919'
library.loc[:,'stageName'] = 'juvenile stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'perfect match'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX25262324,SRP519017,Illumina NovaSeq X Plus,SRS21947445,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,AGSM2024,SAMN42392262,,,,,,,,LH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2024,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX25262323,SRP519017,Illumina NovaSeq X Plus,SRS21947444,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,AGSM2018,SAMN42392261,,,,,,,,EH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2018,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX25262322,SRP519017,Illumina NovaSeq X Plus,SRS21947443,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,AGSM2008,SAMN42392260,,,,,,,,LH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2008,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX25262321,SRP519017,Illumina NovaSeq X Plus,SRS21947442,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,AGSM2007,SAMN42392259,,,,,,,,LH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2007,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX25262320,SRP519017,Illumina NovaSeq X Plus,SRS21947441,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,F,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,AGSF2083,SAMN42392258,,,,,,,,EH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSF2083,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX25262319,SRP519017,Illumina NovaSeq X Plus,SRS21947440,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,F,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,AGSF2073,SAMN42392257,,,,,,,,LH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [9]:
# making these variables because we use them again in the experiment file
my_protocol = 'NEBNext Ultra II Directional RNA Library Prep Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX25262324,SRP519017,Illumina NovaSeq X Plus,SRS21947445,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSM2024,SAMN42392262,,,,,,,,LH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2024,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX25262323,SRP519017,Illumina NovaSeq X Plus,SRS21947444,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSM2018,SAMN42392261,,,,,,,,EH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2018,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX25262322,SRP519017,Illumina NovaSeq X Plus,SRS21947443,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSM2008,SAMN42392260,,,,,,,,LH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2008,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX25262321,SRP519017,Illumina NovaSeq X Plus,SRS21947442,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSM2007,SAMN42392259,,,,,,,,LH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2007,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX25262320,SRP519017,Illumina NovaSeq X Plus,SRS21947441,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,F,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSF2083,SAMN42392258,,,,,,,,EH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSF2083,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX25262319,SRP519017,Illumina NovaSeq X Plus,SRS21947440,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,F,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSF2073,SAMN42392257,,,,,,,,LH,,,17/04/2026,"Preparations of libraries for RNA sequencing was done usi

#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [11]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
library.loc[library["condition"] == "LH", "condition"] = "late hibernation"
library.loc[library["condition"] == "EH", "condition"] = "early hibernation"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX25262324,SRP519017,Illumina NovaSeq X Plus,SRS21947445,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSM2024,SAMN42392262,,,,,,,,late hibernation,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2024,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX25262323,SRP519017,Illumina NovaSeq X Plus,SRS21947444,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSM2018,SAMN42392261,,,,,,,,early hibernation,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2018,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX25262322,SRP519017,Illumina NovaSeq X Plus,SRS21947443,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSM2008,SAMN42392260,,,,,,,,late hibernation,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2008,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX25262321,SRP519017,Illumina NovaSeq X Plus,SRS21947442,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSM2007,SAMN42392259,,,,,,,,late hibernation,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2007,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX25262320,SRP519017,Illumina NovaSeq X Plus,SRS21947441,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,F,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSF2083,SAMN42392258,,,,,,,,early hibernation,,,17/04/2026,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSF2083,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX25262319,SRP519017,Illumina NovaSeq X Plus,SRS21947440,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,F,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSF2073,SAMN42392257,,,,,,,,la

#### annotator id, last modification date

In [12]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-04-17'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX25262324,SRP519017,Illumina NovaSeq X Plus,SRS21947445,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSM2024,SAMN42392262,,,,,,,,late hibernation,,SAC,2026-04-17,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2024,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX25262323,SRP519017,Illumina NovaSeq X Plus,SRS21947444,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSM2018,SAMN42392261,,,,,,,,early hibernation,,SAC,2026-04-17,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2018,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX25262322,SRP519017,Illumina NovaSeq X Plus,SRS21947443,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSM2008,SAMN42392260,,,,,,,,late hibernation,,SAC,2026-04-17,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2008,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX25262321,SRP519017,Illumina NovaSeq X Plus,SRS21947442,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSM2007,SAMN42392259,,,,,,,,late hibernation,,SAC,2026-04-17,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSM2007,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX25262320,SRP519017,Illumina NovaSeq X Plus,SRS21947441,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,F,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSF2083,SAMN42392258,,,,,,,,early hibernation,,SAC,2026-04-17,"Preparations of libraries for RNA sequencing was done using a NEBNext Ultra II Directional Kit, using the polyA mRNA workflow, with each sample multiplexed using the NEBNext Multiplex Oligos workflow. Adapter ligated cDNAs were enriched following 15 PCR cycles.",,AGSF2083,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX25262319,SRP519017,Illumina NovaSeq X Plus,SRS21947440,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,F,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSF2073,SAMN423

#### comments

In [13]:
library.loc[:,'comment'] = 'PMID: 41060812'

#### save complete file with correct columns

In [14]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX25262324,SRP519017,Illumina NovaSeq X Plus,SRS21947445,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSM2024,SAMN42392262,,,,,,,PMID: 41060812,late hibernation,,SAC,2026-04-17
1,SRX25262323,SRP519017,Illumina NovaSeq X Plus,SRS21947444,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSM2018,SAMN42392261,,,,,,,PMID: 41060812,early hibernation,,SAC,2026-04-17
2,SRX25262322,SRP519017,Illumina NovaSeq X Plus,SRS21947443,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSM2008,SAMN42392260,,,,,,,PMID: 41060812,late hibernation,,SAC,2026-04-17
3,SRX25262321,SRP519017,Illumina NovaSeq X Plus,SRS21947442,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSM2007,SAMN42392259,,,,,,,PMID: 41060812,late hibernation,,SAC,2026-04-17
4,SRX25262320,SRP519017,Illumina NovaSeq X Plus,SRS21947441,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,F,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSF2083,SAMN42392258,,,,,,,PMID: 41060812,early hibernation,,SAC,2026-04-17
5,SRX25262319,SRP519017,Illumina NovaSeq X Plus,SRS21947440,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,F,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSF2073,SAMN42392257,,,,,,,PMID: 41060812,late hibernation,,SAC,2026-04-17
6,SRX25262318,SRP519017,Illumina NovaSeq X Plus,SRS21947439,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,F,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSF2060,SAMN42392256,,,,,,,PMID: 41060812,EJ,,SAC,2026-04-17
7,SRX25262317,SRP519017,Illumina NovaSeq X Plus,SRS21947438,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,F,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSF2043,SAMN42392255,,,,,,,PMID: 41060812,late hibernation,,SAC,2026-04-17
8,SRX25262316,SRP519017,Illumina NovaSeq X Plus,SRS21947437,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSM2080,SAMN42392266,,,,,,,PMID: 41060812,early hibernation,,SAC,2026-04-17
9,SRX25262315,SRP519017,Illumina NovaSeq X Plus,SRS21947436,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSM2057,SAMN42392265,,,,,,,PMID: 41060812,early hibernation,,SAC,2026-04-17


### experiment annotations

In [15]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP519017,Arctic ground squirrel arcuate nucleus sequencing,Sex-specific neuroendocrine transcriptomic changes from early to late hibernation in male and female arctic ground squirrels,SRA,,,,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,,PRJNA1134074,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [16]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

14

In [17]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP519017,Arctic ground squirrel arcuate nucleus sequencing,Sex-specific neuroendocrine transcriptomic changes from early to late hibernation in male and female arctic ground squirrels,SRA,total,Bgee 1K,14,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,,PRJNA1134074,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [18]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '41060812'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC12679999/'
experiment.loc[:,'DOI'] = '10.1152/physiolgenomics.00073.2025'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP519017,Arctic ground squirrel arcuate nucleus sequencing,Sex-specific neuroendocrine transcriptomic changes from early to late hibernation in male and female arctic ground squirrels,SRA,total,Bgee 1K,14,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,,PRJNA1134074,41060812,https://pmc.ncbi.nlm.nih.gov/articles/PMC12679999/,10.1152/physiolgenomics.00073.2025,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [19]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [20]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [21]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [24]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [25]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
61981,SRX4406623,SRP154405,Illumina HiSeq 4000,SRS3560536,UBERON:0001134,skeletal muscle tissue,UBERON:0000113,post-juvenile adult stage,,skeletal muscle,adult,perfect match,not documented,perfect match,M,,,9999,,,polyA,,,Model organism or animal sample from Urocitell...,SAMN09690991,,,,,,,PMID: 32488149,Late torpor,,SAC,2026-04-17
61982,SRX4406622,SRP154405,Illumina HiSeq 4000,SRS3560535,UBERON:0001134,skeletal muscle tissue,UBERON:0000113,post-juvenile adult stage,,skeletal muscle,adult,perfect match,not documented,perfect match,M,,,9999,,,polyA,,,Model organism or animal sample from Urocitell...,SAMN09690992,,,,,,,PMID: 32488149,Summer active,,SAC,2026-04-17
61983,SRX25262324,SRP519017,Illumina NovaSeq X Plus,SRS21947445,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSM2024,SAMN42392262,,,,,,,PMID: 41060812,late hibernation,,SAC,2026-04-17
61984,SRX25262323,SRP519017,Illumina NovaSeq X Plus,SRS21947444,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSM2018,SAMN42392261,,,,,,,PMID: 41060812,early hibernation,,SAC,2026-04-17
61985,SRX25262322,SRP519017,Illumina NovaSeq X Plus,SRS21947443,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSM2008,SAMN42392260,,,,,,,PMID: 41060812,late hibernation,,SAC,2026-04-17
61986,SRX25262321,SRP519017,Illumina NovaSeq X Plus,SRS21947442,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,M,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSM2007,SAMN42392259,,,,,,,PMID: 41060812,late hibernation,,SAC,2026-04-17
61987,SRX25262320,SRP519017,Illumina NovaSeq X Plus,SRS21947441,UBERON:0001898,hypothalamus,UBERON:0034919,juvenile stage,,Brain - hypothalamus,juvenile,perfect match,not documented,perfect match,F,,,9999,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,AGSF2083,SAMN42392258,,,,,,,PMID: 41060812,early hibernation,,SAC,2026-04-17


In [26]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1189,SRP316882,Balaenoptera musculus (Blue Whale) and Balaeno...,Blue whales are the largest animals that have ...,SRA,partial,Bgee 1K,1,,,,PRJNA704862,,https://link.springer.com/article/10.1007/s105...,10.1007/s10592-023-01584-5,,"removed DNA samples, no PMID"
1190,SRP154405,"Muscle atrophy, osteoporosis prevention in hib...",Physical inactivity decreases the mechanical l...,SRA,total,Bgee 1K,12,,,,PRJNA481834,32488149,https://pmc.ncbi.nlm.nih.gov/articles/PMC7265340/,10.1038/s41598-020-66030-9,,
1191,SRP519017,Arctic ground squirrel arcuate nucleus sequencing,Sex-specific neuroendocrine transcriptomic cha...,SRA,total,Bgee 1K,14,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,,PRJNA1134074,41060812,https://pmc.ncbi.nlm.nih.gov/articles/PMC12679...,10.1152/physiolgenomics.00073.2025,,


### add annotations to git

In [27]:
! git pull

Already up to date.


In [28]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [29]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [30]:
! git add $git_experiment_path $git_library_path

In [31]:
! git commit -m $commit_message_exp

[develop 9c4032e] adding annotated bulk experiment SRP519017
 2 files changed, 15 insertions(+)


In [32]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.18 KiB | 2.18 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   c799be6..9c4032e  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push